# Overture — Transportation network (live)

Fetch routable road segments for a block and inspect their road classes. Transportation is heavily OpenStreetMap-derived, so expect `ODbL-1.0` rows.

## Setup

Consolidate the imports up front: `EarthLens` is the unified entry point, and `geopandas` reads the GeoParquet that the Overture backend writes to disk.

In [ ]:
import geopandas as gpd
from earthlens.earthlens import EarthLens

### Define the area of interest

A small Times Square block, given as south/north latitude and west/east longitude bounds, plus the output directory for the downloaded GeoParquet.

In [ ]:
LAT_LIM = [40.757, 40.759]  # [south, north]
LON_LIM = [-73.987, -73.984]  # [west, east] — a Times Square block
OUT = "_overture_out"

### Download the transportation segments

Build the `EarthLens` request for the `transportation` theme, then call `download()` on its own line so the construction and the fetch read as separate steps. The Overture backend writes a GeoParquet file and returns the list of written paths.

In [ ]:
overture = EarthLens(
    data_source="overture",
    dataset="transportation",
    variables=["segment"],
    aoi=[LON_LIM[0], LAT_LIM[0], LON_LIM[1], LAT_LIM[1]],
    path=OUT,
)
paths = overture.download()
paths

### Inspect the segments

Read the written GeoParquet into a `GeoDataFrame` and peek at a few rows. Each segment carries a `subtype`, a road `class`, and a per-row `license_id`.

In [ ]:
gdf = gpd.read_parquet(paths[0])
print(len(gdf), 'segments;  geom', gdf.geometry.geom_type.unique())
gdf[['id', 'subtype', 'class', 'license_id']].head()

Road segments by `class`:

In [ ]:
gdf['class'].value_counts()

### Plot the road network

Draw every segment as a line to see the block's road and rail layout.

In [ ]:
ax = gdf.plot(figsize=(5, 5), linewidth=0.8)
ax.set_title('Overture road segments')
ax.set_axis_off()